# Experimentos principales registrados en Comet

Este notebook reconstruye, desde los datos crudos, los dos experimentos que más influyeron en la solución final: el uso de `sample_weight` y la política de decisión basada en el threshold global OOF.

La idea es que sea ejecutable fuera del repositorio original. Si faltan los datos, los descarga desde Kaggle usando `kaggle.json`. Si se activa el registro en Comet, toma la API key desde `.comet.config`.


## Control de registro en Comet

Controlamos si se registran o no los experimentos en Comet. Por defecto queda desactivado para que el notebook pueda ejecutarse sin subir nada. Si se cambia `LOGUEAR_A_COMET` a `True`, se lee `.comet.config` desde el home y se crean experimentos con nombre `final.<nombre_del_experimento>`.

El archivo `.comet.config` puede contener solo la API key o puede venir en formato INI. Antes de importar `comet_ml`, el notebook lo normaliza a un formato que el SDK puede leer para evitar errores de parseo.


In [ ]:
LOGUEAR_A_COMET = False
COMET_CONFIG_PATH = "~/.comet.config"
COMET_PROJECT_NAME = "proyecto-1-taa"
COMET_WORKSPACE = "equipo-a"

## Librerías

Se instalan y cargan las librerías necesarias para descargar datos, preparar tablas, entrenar XGBoost, calcular métricas y, opcionalmente, registrar resultados en Comet.

In [2]:
!pip install numpy pandas scikit-learn xgboost kaggle comet-ml

import json
import math
import os
import shutil
import configparser
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import RobustScaler
from xgboost import XGBClassifier

Defaulting to user installation because normal site-packages is not writeable


## Datos y conexión opcional a Comet

Este bloque prepara lo necesario para trabajar de forma autocontenida. Si `training.csv` y `test.csv` no están en el directorio del notebook, se descargan desde Kaggle. Si se activa Comet, se crea un experimento usando la API key guardada en `.comet.config`.

In [3]:
COMPETITION = "higgs-boson"
RESULTS_DIR = Path("resultados_experimentos_comet")
RESULTS_DIR.mkdir(exist_ok=True)


def notebook_work_dir() -> Path:
    return Path.cwd().resolve()


def request_file(file_name: str, destination: Path, prompt_text: str) -> Path:
    print(prompt_text)
    try:
        from google.colab import files  # type: ignore[import-not-found]
    except ImportError:
        raw_path = input(f"Ingrese la ruta completa al archivo {file_name}: ").strip().strip('"').strip("'")
        if not raw_path:
            raise FileNotFoundError(f"No se indicó una ruta para {file_name}.")
        source = Path(raw_path).expanduser().resolve()
        if not source.exists():
            raise FileNotFoundError(f"No se encontró el archivo indicado: {source}")
        if source != destination.resolve():
            shutil.copy2(source, destination)
    else:
        uploaded = files.upload()
        if file_name not in uploaded:
            raise FileNotFoundError(f"Debe subir un archivo llamado {file_name}.")
        destination.write_bytes(uploaded[file_name])
    return destination


def configure_kaggle_credentials(work_dir: Path) -> None:
    kaggle_file = work_dir / "kaggle.json"
    if not kaggle_file.exists():
        request_file(
            "kaggle.json",
            kaggle_file,
            "No se encontró kaggle.json junto al notebook. Descargue el token desde Kaggle: Account > API > Create New Token.",
        )
    if os.name != "nt":
        kaggle_file.chmod(0o600)
    os.environ["KAGGLE_CONFIG_DIR"] = str(work_dir)


def run_command(command: list[str]) -> None:
    completed = subprocess.run(command, check=False, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.returncode != 0:
        if completed.stderr:
            print(completed.stderr)
        raise RuntimeError(f"Falló el comando: {' '.join(command)}")


def extract_zip(zip_path: Path, output_dir: Path) -> None:
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(output_dir)


def ensure_higgs_data(work_dir: Path) -> tuple[Path, Path]:
    train_path = work_dir / "training.csv"
    test_path = work_dir / "test.csv"
    if train_path.exists() and test_path.exists():
        print("Datos encontrados en el directorio actual.")
        return train_path, test_path

    configure_kaggle_credentials(work_dir)
    competition_zip = work_dir / f"{COMPETITION}.zip"
    training_zip = work_dir / "training.zip"
    test_zip = work_dir / "test.zip"

    if not competition_zip.exists() and not (training_zip.exists() and test_zip.exists()):
        kaggle_executable = shutil.which("kaggle")
        if kaggle_executable is None:
            raise RuntimeError("No se encontró el ejecutable kaggle.")
        run_command([kaggle_executable, "competitions", "download", "-c", COMPETITION, "-p", str(work_dir)])

    if competition_zip.exists() and not (training_zip.exists() and test_zip.exists()):
        extract_zip(competition_zip, work_dir)
    if not train_path.exists():
        extract_zip(training_zip, work_dir)
    if not test_path.exists():
        extract_zip(test_zip, work_dir)

    if not train_path.exists() or not test_path.exists():
        raise FileNotFoundError("No se pudieron preparar training.csv y test.csv.")
    return train_path, test_path



def parse_comet_api_key(config_text: str) -> str:
    text = config_text.strip()
    if not text:
        raise ValueError("La API key de Comet está vacía.")
    if text.startswith("[") or "=" in text:
        parser = configparser.ConfigParser()
        parser.read_string(text)
        for section in parser.sections():
            if parser.has_option(section, "api_key"):
                return parser.get(section, "api_key").strip()
        raise ValueError("No se encontró api_key dentro de .comet.config.")
    return text.splitlines()[0].strip()


def comet_ini_text(api_key: str) -> str:
    return (
        "[comet]\n"
        f"api_key = {api_key}\n"
        f"project_name = {COMET_PROJECT_NAME}\n"
        f"workspace = {COMET_WORKSPACE}\n"
    )


def normalize_comet_config_files(api_key: str, primary_path: Path) -> None:
    candidate_paths = [
        primary_path,
        Path.cwd() / ".comet.config",
        Path("/content/drive/MyDrive/.comet.config"),
    ]
    normalized = comet_ini_text(api_key)
    for candidate in candidate_paths:
        candidate = candidate.expanduser()
        if not candidate.exists():
            continue
        current_text = candidate.read_text().strip()
        if current_text == normalized.strip():
            continue
        # El SDK de Comet intenta parsear estos archivos automáticamente como INI.
        # Si contienen solo la key cruda, falla antes de crear el experimento.
        candidate.write_text(normalized)


def read_comet_api_key() -> str:
    config_path = Path(COMET_CONFIG_PATH).expanduser()
    if not config_path.exists():
        request_file(
            ".comet.config",
            config_path,
            "No se encontró .comet.config en el home. Suba o indique el archivo con la API key de Comet.",
        )
    api_key = parse_comet_api_key(config_path.read_text())
    normalize_comet_config_files(api_key, config_path)
    return api_key

def create_comet_experiment(experiment_name: str, tags: list[str] | None = None):
    if not LOGUEAR_A_COMET:
        return None
    api_key = read_comet_api_key()
    from comet_ml import Experiment

    experiment = Experiment(
        api_key=api_key,
        project_name=COMET_PROJECT_NAME,
        workspace=COMET_WORKSPACE,
    )
    experiment.set_name(f"final.{experiment_name}")
    for tag in tags or []:
        experiment.add_tag(tag)
    return experiment


def log_dataframe_asset(experiment, df: pd.DataFrame, file_name: str) -> None:
    if experiment is None:
        return
    path = RESULTS_DIR / file_name
    df.to_csv(path, index=False)
    experiment.log_asset(str(path), file_name=file_name)

## Configuración común del modelo

Se fija el mismo preprocesamiento `P04` y la misma familia de modelo usada en el informe. `Weight` no entra como feature: en el primer experimento se compara explícitamente qué pasa cuando se usa o no se usa como `sample_weight`.

In [4]:
SEED = 42
N_SPLITS = 5
VALID_SIZE = 0.1
MISSING_MARKER = -999.0
EPS = 1e-6
JET_NUM_COLUMN = "PRI_jet_num"
JET_NUM_VALUES = [0, 1, 2, 3]
FIXED_JET_ONE_HOT_COLUMNS = [f"PRI_jet_num_{value}" for value in JET_NUM_VALUES]
SPECIAL_COLUMNS = ["EventId", "Weight", "Label", "KaggleSet", "KaggleWeight"]

ALLOWED_FEATURE_COLUMNS = [
    "DER_mass_MMC", "DER_mass_transverse_met_lep", "DER_mass_vis", "DER_pt_h",
    "DER_deltaeta_jet_jet", "DER_mass_jet_jet", "DER_prodeta_jet_jet",
    "DER_deltar_tau_lep", "DER_pt_tot", "DER_sum_pt", "DER_pt_ratio_lep_tau",
    "DER_met_phi_centrality", "DER_lep_eta_centrality", "PRI_tau_pt", "PRI_tau_eta",
    "PRI_tau_phi", "PRI_lep_pt", "PRI_lep_eta", "PRI_lep_phi", "PRI_met",
    "PRI_met_phi", "PRI_met_sumet", "PRI_jet_num", "PRI_jet_leading_pt",
    "PRI_jet_leading_eta", "PRI_jet_leading_phi", "PRI_jet_subleading_pt",
    "PRI_jet_subleading_eta", "PRI_jet_subleading_phi", "PRI_jet_all_pt",
]

BASE_LOG_COLUMNS = [
    "DER_mass_MMC", "DER_mass_transverse_met_lep", "DER_mass_vis", "DER_pt_h",
    "DER_deltaeta_jet_jet", "DER_mass_jet_jet", "DER_deltar_tau_lep", "DER_pt_tot",
    "DER_sum_pt", "DER_pt_ratio_lep_tau", "PRI_tau_pt", "PRI_lep_pt", "PRI_met",
    "PRI_met_sumet", "PRI_jet_leading_pt", "PRI_jet_subleading_pt", "PRI_jet_all_pt",
]
P04_ENGINEERED_LOG_COLUMNS = ["FE_tau_lep_pt_sum"]

BASE_XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": SEED,
    "tree_method": "hist",
    "n_jobs": -1,
    "missing": np.nan,
    "learning_rate": 0.03,
    "n_estimators": 800,
    "max_depth": 6,
    "min_child_weight": 1,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "scale_pos_weight": 1.5346,
    "gamma": 0.0,
    "max_delta_step": 0,
}

SAMPLE_WEIGHT_CONFIGS = [
    {"config_id": "R04_sin_sample_weight", "sample_weight_used": False, "reg_alpha": 0.1, "reg_lambda": 1.0},
    {"config_id": "W03_con_sample_weight", "sample_weight_used": True, "reg_alpha": 0.5, "reg_lambda": 1.0},
    {"config_id": "SW04_final", "sample_weight_used": True, "reg_alpha": 0.5, "reg_lambda": 2.0},
]

## Preprocesamiento P04 y métricas

Este bloque define las transformaciones físicas de `P04` y las funciones para calcular AMS, precision, recall y F1. La métrica central es AMS, porque pondera cuánto signal se selecciona y cuánto background ponderado se contamina.

In [5]:
def abs_delta_phi(phi_a: pd.Series, phi_b: pd.Series) -> pd.Series:
    return ((phi_a - phi_b + np.pi) % (2 * np.pi) - np.pi).abs()


def add_p04_features(X: pd.DataFrame) -> pd.DataFrame:
    result = X.copy()
    result["FE_abs_dphi_tau_lep"] = abs_delta_phi(result["PRI_tau_phi"], result["PRI_lep_phi"])
    result["FE_abs_dphi_tau_met"] = abs_delta_phi(result["PRI_tau_phi"], result["PRI_met_phi"])
    result["FE_abs_dphi_lep_met"] = abs_delta_phi(result["PRI_lep_phi"], result["PRI_met_phi"])
    result["FE_abs_deta_tau_lep"] = (result["PRI_tau_eta"] - result["PRI_lep_eta"]).abs()
    result["FE_tau_lep_pt_sum"] = result["PRI_tau_pt"] + result["PRI_lep_pt"]
    result["FE_tau_pt_fraction"] = result["PRI_tau_pt"] / (result["PRI_tau_pt"] + result["PRI_lep_pt"] + EPS)
    result["FE_met_over_tau_lep_pt"] = result["PRI_met"] / (result["PRI_tau_pt"] + result["PRI_lep_pt"] + EPS)
    result["FE_met_over_sumet"] = result["PRI_met"] / (result["PRI_met_sumet"] + EPS)
    result["FE_pt_balance"] = result["DER_pt_tot"] / (result["DER_sum_pt"] + EPS)
    result["FE_mmc_over_visible_mass"] = result["DER_mass_MMC"] / (result["DER_mass_vis"] + EPS)
    result["FE_transverse_mass_over_visible_mass"] = result["DER_mass_transverse_met_lep"] / (result["DER_mass_vis"] + EPS)
    result["FE_jet_all_pt_fraction"] = result["PRI_jet_all_pt"] / (result["DER_sum_pt"] + EPS)
    result["FE_leading_jet_pt_fraction"] = result["PRI_jet_leading_pt"] / (result["PRI_jet_all_pt"] + EPS)
    result["FE_subleading_jet_pt_fraction"] = result["PRI_jet_subleading_pt"] / (result["PRI_jet_all_pt"] + EPS)
    result["FE_jet_pt_balance"] = (result["PRI_jet_leading_pt"] - result["PRI_jet_subleading_pt"]) / (
        result["PRI_jet_leading_pt"] + result["PRI_jet_subleading_pt"] + EPS
    )
    result["FE_abs_dphi_leading_subleading_jet"] = abs_delta_phi(result["PRI_jet_leading_phi"], result["PRI_jet_subleading_phi"])
    result["FE_mass_jet_over_sum_pt"] = result["DER_mass_jet_jet"] / (result["DER_sum_pt"] + EPS)
    return result


class P04Preprocessor(BaseEstimator, TransformerMixin):
    def fit(self, X: pd.DataFrame, y: pd.Series | None = None) -> "P04Preprocessor":
        prepared = self._prepare_features(X)
        self.missing_indicator_cols_ = [
            column for column in prepared.columns
            if column != JET_NUM_COLUMN and prepared[column].isna().any()
        ]
        continuous = prepared.drop(columns=[JET_NUM_COLUMN])
        self.continuous_cols_ = continuous.columns.tolist()
        self.log_cols_ = [
            column for column in BASE_LOG_COLUMNS + P04_ENGINEERED_LOG_COLUMNS
            if column in self.continuous_cols_
        ]
        logged = self._apply_log1p(continuous)
        self.imputer_ = SimpleImputer(strategy="median")
        imputed = self.imputer_.fit_transform(logged[self.continuous_cols_])
        self.scaler_ = RobustScaler(with_centering=True, with_scaling=True, quantile_range=(25.0, 75.0))
        self.scaler_.fit(imputed)
        self.final_feature_order_ = (
            self.continuous_cols_
            + [f"is_missing_{column}" for column in self.missing_indicator_cols_]
            + FIXED_JET_ONE_HOT_COLUMNS
        )
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        prepared = self._prepare_features(X)
        indicator_frame = pd.DataFrame(index=prepared.index)
        for column in self.missing_indicator_cols_:
            indicator_frame[f"is_missing_{column}"] = prepared[column].isna().astype(np.float32)
        jet_one_hot = pd.DataFrame(
            {f"PRI_jet_num_{value}": (prepared[JET_NUM_COLUMN].astype(int) == value).astype(np.float32) for value in JET_NUM_VALUES},
            index=prepared.index,
        )
        continuous = prepared.drop(columns=[JET_NUM_COLUMN])[self.continuous_cols_]
        logged = self._apply_log1p(continuous)
        imputed = self.imputer_.transform(logged[self.continuous_cols_])
        scaled = self.scaler_.transform(imputed)
        scaled_frame = pd.DataFrame(scaled, columns=self.continuous_cols_, index=prepared.index)
        result = pd.concat([scaled_frame, indicator_frame, jet_one_hot], axis=1)
        return result[self.final_feature_order_].astype(np.float32)

    def _prepare_features(self, X: pd.DataFrame) -> pd.DataFrame:
        if JET_NUM_COLUMN not in X.columns:
            raise ValueError("PRI_jet_num no existe antes del preprocesamiento.")
        return add_p04_features(X)

    def _apply_log1p(self, X: pd.DataFrame) -> pd.DataFrame:
        result = X.copy()
        for column in self.log_cols_:
            values = result[column].dropna()
            if (values < 0).any():
                raise ValueError(f"{column} tiene valores negativos y no admite log1p.")
            result[column] = np.log1p(result[column])
        return result


def ams_from_weights(signal_weight: float, background_weight: float) -> float:
    br = 10.0
    radicand = 2.0 * ((signal_weight + background_weight + br) * math.log(1.0 + signal_weight / (background_weight + br)) - signal_weight)
    return math.sqrt(max(radicand, 0.0))


def metrics_for_predictions(y_true: np.ndarray, weights: np.ndarray, predictions: np.ndarray) -> dict[str, float]:
    signal_weight = float(weights[(y_true == 1) & (predictions == 1)].sum())
    background_weight = float(weights[(y_true == 0) & (predictions == 1)].sum())
    return {
        "ams": ams_from_weights(signal_weight, background_weight),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "signal_weight": signal_weight,
        "background_weight": background_weight,
        "signal_rate": float(predictions.mean()),
        "signal_percent": float(predictions.mean() * 100.0),
    }


def best_ams_by_ranking(scores: np.ndarray, y_true: np.ndarray, weights: np.ndarray) -> dict[str, float | np.ndarray]:
    order = np.argsort(scores)[::-1]
    sorted_y = y_true[order]
    sorted_w = weights[order]
    signal_cumsum = np.cumsum(sorted_w * (sorted_y == 1))
    background_cumsum = np.cumsum(sorted_w * (sorted_y == 0))
    ams_values = np.array([ams_from_weights(float(s), float(b)) for s, b in zip(signal_cumsum, background_cumsum)])
    best_position = int(np.argmax(ams_values))
    predictions = np.zeros(len(scores), dtype=np.int8)
    predictions[order[: best_position + 1]] = 1
    threshold = float(scores[order[best_position]])
    metrics = metrics_for_predictions(y_true, weights, predictions)
    metrics.update({
        "threshold": threshold,
        "selected_count": int(best_position + 1),
        "best_rank_position": int(best_position + 1),
        "predictions": predictions,
    })
    return metrics


def top_percent_predictions(scores: np.ndarray, percent: float) -> np.ndarray:
    n_signal = int(np.ceil(len(scores) * percent / 100.0))
    order = np.argsort(scores)[::-1]
    predictions = np.zeros(len(scores), dtype=np.int8)
    predictions[order[:n_signal]] = 1
    return predictions

## Carga y preparación del dataset

Se cargan los datos oficiales y se reconstruye el preprocesamiento `P04` desde cero. Para mantener el mismo esquema usado en el pipeline final, se ajusta el preprocesador en un split interno y luego se trabaja con la matriz procesada completa.

In [6]:
def validate_raw_higgs(train_df: pd.DataFrame, test_df: pd.DataFrame) -> None:
    feature_cols = [column for column in train_df.columns if column not in SPECIAL_COLUMNS]
    test_feature_cols = [column for column in test_df.columns if column not in SPECIAL_COLUMNS]
    if feature_cols != ALLOWED_FEATURE_COLUMNS:
        raise ValueError("Las features de training.csv no coinciden con la especificación esperada.")
    if test_feature_cols != feature_cols:
        raise ValueError("Las features de training.csv y test.csv no coinciden.")


def build_processed_p04(train_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series, pd.Series, pd.Series]:
    X_full = train_df[ALLOWED_FEATURE_COLUMNS].replace(MISSING_MARKER, np.nan)
    y_full = train_df["Label"].map({"b": 0, "s": 1}).astype(np.int8)
    weight_full = train_df["Weight"].astype(np.float64)
    event_id_full = train_df["EventId"].astype(np.int64)

    train_idx, valid_idx = train_test_split(
        X_full.index.to_numpy(),
        test_size=VALID_SIZE,
        random_state=SEED,
        stratify=y_full,
    )

    X_train_raw = X_full.loc[train_idx].reset_index(drop=True)
    X_valid_raw = X_full.loc[valid_idx].reset_index(drop=True)
    preprocessor = P04Preprocessor().fit(X_train_raw)

    X_train_processed = preprocessor.transform(X_train_raw)
    X_valid_processed = preprocessor.transform(X_valid_raw)

    X_model = pd.concat([X_train_processed, X_valid_processed], ignore_index=True)
    y_model = pd.concat([y_full.loc[train_idx], y_full.loc[valid_idx]], ignore_index=True)
    w_model = pd.concat([weight_full.loc[train_idx], weight_full.loc[valid_idx]], ignore_index=True)
    event_id_model = pd.concat([event_id_full.loc[train_idx], event_id_full.loc[valid_idx]], ignore_index=True)
    return X_model, y_model, w_model, event_id_model


work_dir = notebook_work_dir()
train_path, test_path = ensure_higgs_data(work_dir)
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
validate_raw_higgs(train_df, test_df)

X_model, y_model, w_model, event_id_model = build_processed_p04(train_df)
print("X_model:", X_model.shape)
print("Signal rate train:", float(y_model.mean()))

Datos encontrados en el directorio actual.
X_model: (250000, 67)
Signal rate train: 0.342668


## Experimento 1: efecto de `sample_weight`

Este experimento muestra el cambio más importante del proceso. Se comparan configuraciones sin pesos de entrenamiento, con pesos, y la configuración final `SW04` (Sample Weight - Preprocesamiento 04). La salida principal es una tabla donde se ve cómo cambia el AMS medio en validación cruzada.

In [7]:
def run_cv_experiment(config: dict[str, object]) -> tuple[pd.DataFrame, dict[str, object], pd.DataFrame]:
    params = {**BASE_XGB_PARAMS, "reg_alpha": config["reg_alpha"], "reg_lambda": config["reg_lambda"]}
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    fold_rows = []
    oof_scores = np.zeros(len(X_model), dtype=np.float64)

    y_array = y_model.to_numpy(dtype=np.int8)
    w_array = w_model.to_numpy(dtype=np.float64)

    for fold_id, (train_idx, valid_idx) in enumerate(cv.split(X_model, y_model), start=1):
        model = XGBClassifier(**params)
        fit_kwargs = {}
        if config["sample_weight_used"]:
            fit_kwargs["sample_weight"] = w_model.iloc[train_idx].reset_index(drop=True)

        start = time.perf_counter()
        model.fit(
            X_model.iloc[train_idx].reset_index(drop=True),
            y_model.iloc[train_idx].reset_index(drop=True),
            **fit_kwargs,
        )
        fit_seconds = time.perf_counter() - start
        if model.classes_.tolist() != [0, 1]:
            raise ValueError(f"Orden inesperado de clases: {model.classes_.tolist()}")

        scores = model.predict_proba(X_model.iloc[valid_idx])[:, 1]
        oof_scores[valid_idx] = scores
        best_metrics = best_ams_by_ranking(scores, y_array[valid_idx], w_array[valid_idx])
        fold_rows.append({
            "config_id": config["config_id"],
            "fold_id": fold_id,
            "sample_weight_used": bool(config["sample_weight_used"]),
            "reg_alpha": config["reg_alpha"],
            "reg_lambda": config["reg_lambda"],
            "ams": best_metrics["ams"],
            "threshold": best_metrics["threshold"],
            "precision": best_metrics["precision"],
            "recall": best_metrics["recall"],
            "f1": best_metrics["f1"],
            "signal_percent": best_metrics["signal_percent"],
            "fit_seconds": fit_seconds,
        })

    fold_df = pd.DataFrame(fold_rows)
    summary = {
        "config_id": config["config_id"],
        "sample_weight_used": bool(config["sample_weight_used"]),
        "reg_alpha": config["reg_alpha"],
        "reg_lambda": config["reg_lambda"],
        "ams_mean": float(fold_df["ams"].mean()),
        "ams_std": float(fold_df["ams"].std(ddof=1)),
        "threshold_median": float(fold_df["threshold"].median()),
        "precision_mean": float(fold_df["precision"].mean()),
        "recall_mean": float(fold_df["recall"].mean()),
        "f1_mean": float(fold_df["f1"].mean()),
        "signal_percent_mean": float(fold_df["signal_percent"].mean()),
        "fit_seconds_mean": float(fold_df["fit_seconds"].mean()),
    }
    oof_df = pd.DataFrame({
        "EventId": event_id_model.to_numpy(dtype=np.int64),
        "y_true": y_model.to_numpy(dtype=np.int8),
        "weight": w_model.to_numpy(dtype=np.float64),
        "score": oof_scores,
        "config_id": config["config_id"],
    })
    return fold_df, summary, oof_df


all_fold_metrics = []
summary_rows = []
sw04_oof_df = None

for config in SAMPLE_WEIGHT_CONFIGS:
    fold_df, summary, oof_df = run_cv_experiment(config)
    all_fold_metrics.append(fold_df)
    summary_rows.append(summary)
    if config["config_id"] == "SW04_final":
        sw04_oof_df = oof_df

sample_weight_fold_df = pd.concat(all_fold_metrics, ignore_index=True)
sample_weight_summary_df = pd.DataFrame(summary_rows).sort_values("ams_mean", ascending=False)

sample_weight_fold_df.to_csv(RESULTS_DIR / "final_sample_weight_fold_metrics.csv", index=False)
sample_weight_summary_df.to_csv(RESULTS_DIR / "final_sample_weight_summary.csv", index=False)
if sw04_oof_df is not None:
    sw04_oof_df.to_csv(RESULTS_DIR / "final_sw04_oof_scores.csv", index=False)

display(sample_weight_summary_df)

,config_id,sample_weight_used,reg_alpha,reg_lambda,ams_mean,ams_std,threshold_median,precision_mean,recall_mean,f1_mean,signal_percent_mean,fit_seconds_mean
2,SW04_final,True,0.5,2.0,1.670944,0.078382,0.054097,0.911956,0.409750,0.564003,15.4196,15.081189
1,W03_con_sample_weight,True,0.5,1.0,1.663999,0.073962,0.062080,0.914117,0.402746,0.558438,15.1084,13.528199
0,R04_sin_sample_weight,False,0.1,1.0,1.635711,0.093489,0.886797,0.932409,0.404648,0.563448,14.8876,15.404902


## Registro Comet del experimento 1

Si `LOGUEAR_A_COMET=True`, se sube a Comet un experimento llamado `final.sample_weight`. Se registran los hiperparámetros, el mejor AMS medio y las tablas con métricas por fold y resumen.

In [8]:
sample_weight_experiment = create_comet_experiment("sample_weight", tags=["final", "sample_weight", "P04", "XGBoost"])
if sample_weight_experiment is not None:
    sample_weight_experiment.log_parameters({
        "pipeline": "P04",
        "model": "XGBoost",
        "n_splits": N_SPLITS,
        "seed": SEED,
        "configs": json.dumps(SAMPLE_WEIGHT_CONFIGS, ensure_ascii=False),
    })
    best_row = sample_weight_summary_df.iloc[0]
    sample_weight_experiment.log_metric("best_ams_mean", best_row["ams_mean"])
    sample_weight_experiment.log_metric("best_ams_std", best_row["ams_std"])
    sample_weight_experiment.log_metric("best_recall_mean", best_row["recall_mean"])
    sample_weight_experiment.log_metric("best_precision_mean", best_row["precision_mean"])
    for _, row in sample_weight_summary_df.iterrows():
        prefix = str(row["config_id"])
        sample_weight_experiment.log_metric(f"{prefix}_ams_mean", row["ams_mean"])
        sample_weight_experiment.log_metric(f"{prefix}_ams_std", row["ams_std"])
    log_dataframe_asset(sample_weight_experiment, sample_weight_summary_df, "final_sample_weight_summary.csv")
    log_dataframe_asset(sample_weight_experiment, sample_weight_fold_df, "final_sample_weight_fold_metrics.csv")
    sample_weight_experiment.end()

/usr/lib/python3/dist-packages/requests/__init__.py:87: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (4.0.0) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({}) doesn't match a supported "
COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: xgboost, sklearn.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/equipo-a/proyecto-1-taa/c831d1f2f7e24522bc0089f10821de11

COMET INFO: Couldn't find a Git repository in '/home/santig14/fing/taa/entrega_taller_1' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: --------------------------

## Experimento 2: política de decisión por threshold global OOF

Con el modelo final ya fijado (`SW04`), se estudia cómo transformar scores en clases. Se comparan varias reglas sobre los scores OOF. La política final elegida es `global_oof_optimal_threshold`: se busca sobre los scores out-of-fold el threshold que maximiza AMS y luego se usa ese valor absoluto para clasificar los eventos.


In [9]:
def evaluate_threshold_policy(policy_id: str, scores: np.ndarray, y_true: np.ndarray, weights: np.ndarray, rule: str, value: float | None = None) -> dict[str, float | str]:
    if rule == "global_best":
        best = best_ams_by_ranking(scores, y_true, weights)
        predictions = best["predictions"]
        threshold = best["threshold"]
    elif rule == "top_percent":
        predictions = top_percent_predictions(scores, float(value))
        selected_scores = scores[predictions == 1]
        threshold = float(selected_scores.min()) if len(selected_scores) else np.nan
    elif rule == "threshold":
        threshold = float(value)
        predictions = (scores >= threshold).astype(np.int8)
    else:
        raise ValueError(f"Regla desconocida: {rule}")

    metrics = metrics_for_predictions(y_true, weights, predictions)
    return {
        "policy_id": policy_id,
        "rule": rule,
        "value": value,
        "threshold": threshold,
        **metrics,
        "predicted_signal_count": int(predictions.sum()),
        "predicted_background_count": int(len(predictions) - predictions.sum()),
    }


if sw04_oof_df is None:
    raise RuntimeError("No se encontraron los scores OOF de SW04.")

scores = sw04_oof_df["score"].to_numpy(dtype=np.float64)
y_true = sw04_oof_df["y_true"].to_numpy(dtype=np.int8)
weights = sw04_oof_df["weight"].to_numpy(dtype=np.float64)
sw04_fold_thresholds = sample_weight_fold_df.loc[
    sample_weight_fold_df["config_id"] == "SW04_final", "threshold"
]

threshold_policy_rows = [
    evaluate_threshold_policy("global_oof_optimal_threshold", scores, y_true, weights, "global_best"),
    evaluate_threshold_policy("top_14_percent_signal", scores, y_true, weights, "top_percent", 14.0),
    evaluate_threshold_policy("top_15_percent_signal", scores, y_true, weights, "top_percent", 15.0),
    evaluate_threshold_policy("median_fold_threshold", scores, y_true, weights, "threshold", float(sw04_fold_thresholds.median())),
]

threshold_policy_df = pd.DataFrame(threshold_policy_rows).sort_values("ams", ascending=False)
threshold_policy_df.to_csv(RESULTS_DIR / "final_threshold_policy_comparison.csv", index=False)
display(threshold_policy_df)

,policy_id,rule,value,threshold,ams,precision,recall,f1,signal_weight,background_weight,signal_rate,signal_percent,predicted_signal_count,predicted_background_count
0,global_oof_optimal_threshold,global_best,NaN,0.064023,3.698632,0.921009,0.376329,0.534329,212.084356,3208.082300,0.140016,14.0016,35004,214996
1,top_14_percent_signal,top_percent,14.000000,0.064041,3.697918,0.921000,0.376283,0.534280,212.042928,3208.082300,0.140000,14.0000,35000,215000
2,top_15_percent_signal,top_percent,15.000000,0.056164,3.681684,0.913787,0.400002,0.556432,229.525192,3800.818755,0.150000,15.0000,37500,212500
3,median_fold_threshold,threshold,0.054097,0.054097,3.678157,0.911858,0.406726,0.562537,234.545303,3978.802855,0.152844,15.2844,38211,211789


## Registro Comet del experimento 2

Si se activa Comet, esta celda sube el experimento `final.politica_threshold_global_oof`. Se registra la comparación de políticas y, en particular, las métricas de la política `global_oof_optimal_threshold`.


In [10]:
threshold_experiment = create_comet_experiment(
    "politica_threshold_global_oof",
    tags=["final", "threshold", "global_oof", "P04", "XGBoost"],
)
if threshold_experiment is not None:
    global_oof_row = threshold_policy_df.loc[
        threshold_policy_df["policy_id"] == "global_oof_optimal_threshold"
    ].iloc[0]
    top14_row = threshold_policy_df.loc[threshold_policy_df["policy_id"] == "top_14_percent_signal"].iloc[0]
    threshold_experiment.log_parameters({
        "pipeline": "P04",
        "model": "XGBoost SW04",
        "policy": "global_oof_optimal_threshold",
        "decision_threshold": global_oof_row["threshold"],
        "seed": SEED,
        "n_splits": N_SPLITS,
    })
    threshold_experiment.log_metric("global_oof_ams", global_oof_row["ams"])
    threshold_experiment.log_metric("global_oof_threshold", global_oof_row["threshold"])
    threshold_experiment.log_metric("global_oof_precision", global_oof_row["precision"])
    threshold_experiment.log_metric("global_oof_recall", global_oof_row["recall"])
    threshold_experiment.log_metric("global_oof_f1", global_oof_row["f1"])
    threshold_experiment.log_metric("global_oof_signal_percent", global_oof_row["signal_percent"])
    threshold_experiment.log_metric("top14_ams_reference", top14_row["ams"])
    threshold_experiment.log_metric("top14_signal_percent_reference", top14_row["signal_percent"])
    threshold_experiment.log_metric("best_policy_ams", threshold_policy_df.iloc[0]["ams"])
    log_dataframe_asset(threshold_experiment, threshold_policy_df, "final_threshold_policy_comparison.csv")
    threshold_experiment.end()


COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: xgboost, sklearn.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/equipo-a/proyecto-1-taa/21b7c78a5a764ea183b8b6c026caa5a3

COMET INFO: Couldn't find a Git repository in '/home/santig14/fing/taa/entrega_taller_1' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : final.politica_threshold_global_oof
COMET INFO:     url        

## Resumen final

Estas tablas dejan trazado el camino de los dos aportes principales: primero el salto producido por `sample_weight`, y luego la decisión de usar el threshold global OOF como política final de clasificación.


In [11]:
display(sample_weight_summary_df[["config_id", "sample_weight_used", "reg_alpha", "reg_lambda", "ams_mean", "ams_std", "threshold_median"]])
display(threshold_policy_df[["policy_id", "ams", "signal_percent", "precision", "recall", "f1"]])
print(f"Resultados guardados en: {RESULTS_DIR.resolve()}")

,config_id,sample_weight_used,reg_alpha,reg_lambda,ams_mean,ams_std,threshold_median
2,SW04_final,True,0.5,2.0,1.670944,0.078382,0.054097
1,W03_con_sample_weight,True,0.5,1.0,1.663999,0.073962,0.062080
0,R04_sin_sample_weight,False,0.1,1.0,1.635711,0.093489,0.886797


,policy_id,ams,signal_percent,precision,recall,f1
0,global_oof_optimal_threshold,3.698632,14.0016,0.921009,0.376329,0.534329
1,top_14_percent_signal,3.697918,14.0000,0.921000,0.376283,0.534280
2,top_15_percent_signal,3.681684,15.0000,0.913787,0.400002,0.556432
3,median_fold_threshold,3.678157,15.2844,0.911858,0.406726,0.562537


Resultados guardados en: /home/santig14/fing/taa/entrega_taller_1/resultados_experimentos_comet
